## Downloading the BCSS dataset

### The Breast Cancer Semantic Segmentation dataset

#### Getting the metadata

In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"  # PyTorch for GPU

In [2]:
import pathlib

if not pathlib.Path("bcss_repo").exists():
    !git clone --depth 1 https://github.com/PathologyDataScience/BCSS.git bcss_repo

In [3]:
import pandas as pd

pd.read_csv("bcss_repo/meta/gtruth_codes.tsv", sep="\t")

,label,GT_code
0,outside_roi,0
1,tumor,1
2,stroma,2
3,lymphocytic_infiltrate,3
4,necrosis_or_debris,4
5,glandular_secretions,5
6,blood,6
7,exclude,7
8,metaplasia_NOS,8
9,fat,9


#### Downloading the images and masks

In [4]:
import csv
import io
import time

import girder_client
import numpy as np
import requests
from PIL import Image
from skimage.transform import resize

Image.MAX_IMAGE_PIXELS = None

api_url = "https://demo.kitware.com/histomicstk/api/v1/"
source_folder_id = "5bbdeba3e629140048d017bb"
mpp = 0.5  # microns per pixel: half the 0.25 MPP scan resolution

input_dir = pathlib.Path("bcss/images")
target_dir = pathlib.Path("bcss/masks")
input_dir.mkdir(parents=True, exist_ok=True)
target_dir.mkdir(parents=True, exist_ok=True)

In [5]:
def with_retry(fn, tries=4):
    for attempt in range(tries):
        try:
            return fn()
        except Exception as err:
            if attempt == tries - 1:
                raise
            print(f"retry after error: {err}")
            time.sleep(5 * (attempt + 1))

def download_roi(row):
    xmin, ymin, xmax, ymax = (int(float(row[k])) for k in ("xmin", "ymin", "xmax", "ymax"))
    fname = f"{row['']}_xmin{xmin}_ymin{ymin}.png"
    if (input_dir / fname).exists() and (target_dir / fname).exists():
        return
    print("downloading", fname)
    mm = 0.001 * mpp
    region = (f"item/{slide_ids[row[''][:12]]}/tiles/region"
              f"?left={xmin}&right={xmax}&top={ymin}&bottom={ymax}"
              f"&mm_x={mm:.5f}&mm_y={mm:.5f}&encoding=PNG")
    rgb = Image.open(io.BytesIO(with_retry(lambda: gc.get(region, jsonResp=False)).content)).convert("RGB")
    response = with_retry(lambda: requests.get(row["mask_link"], timeout=120))
    response.raise_for_status()
    mask = np.array(Image.open(io.BytesIO(response.content)))
    mask = resize(mask, rgb.size[::-1], order=0, preserve_range=True, anti_aliasing=False)
    rgb.save(input_dir / f"{fname}.tmp.png")
    Image.fromarray(mask.astype("uint8")).save(target_dir / f"{fname}.tmp.png")
    (input_dir / f"{fname}.tmp.png").replace(input_dir / fname)
    (target_dir / f"{fname}.tmp.png").replace(target_dir / fname)

In [6]:
gc = girder_client.GirderClient(apiUrl=api_url)
slide_ids = {item["name"][:12]: item["_id"]
             for item in gc.get(f"item?folderId={source_folder_id}&limit=1000")}

with open("bcss_repo/meta/roiBounds.csv", newline="") as f:
    rows = list(csv.DictReader(f))

for row in rows:
    download_roi(row)

#### Checking the download

In [7]:
input_img_paths = sorted(input_dir.glob("*.png"))
target_paths = sorted(target_dir.glob("*.png"))

mismatches = 0
for input_path, target_path in zip(input_img_paths, target_paths):
    with Image.open(input_path) as img, Image.open(target_path) as target:
        mismatches += img.size != target.size
size_gb = sum(p.stat().st_size for p in input_img_paths + target_paths) / 1e9
print(f"{len(input_img_paths)} images, {len(target_paths)} masks, "
      f"{mismatches} size mismatches, {size_gb:.2f} GB")

151 images, 151 masks, 0 size mismatches, 2.00 GB
